## Import libraries

In [2]:
import os
from glob import glob
import sys
from obspy import Stream, read
from collections import defaultdict

## Script for merging E/N/Z components and save to `*.mseed` files

In [4]:
base_dir="./waveforms" 
output_dir="mseed_merged" 
os.makedirs(output_dir, exist_ok=True)

date_dirs=[d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d)) and d.isdigit()]

for date_dir in date_dirs:
    date_dir_path=os.path.join(base_dir, date_dir)
    print(f"\nProcessing directory: {date_dir_path}")

    station_groups=defaultdict(lambda: {'E': None, 'N': None, 'Z': None})
    
    all_files_in_date_dir=glob(os.path.join(date_dir_path, "KS.*"))

    for f_path in all_files_in_date_dir:
        try:
            filename=os.path.basename(f_path)
            parts=filename.split('.')
            # 'KS.CHOA.HGE.2025.305.00.00.00'
            net=parts[0]       # KS
            sta=parts[1]       # CHOA
            chan=parts[2]      # HGE, HHN, ELZ
            
            band_instr=chan[0:2] # HG, HH, EL
            component=chan[2]    # E, N, Z
                
            group_key=(net, sta, band_instr)
            station_groups[group_key][component]=f_path
            
        except Exception as e:
            print(f"Skip {filename} (Error: {e})")
    
    for (net, sta, band_instr), components in station_groups.items():
        
        file_e=components.get('E')
        file_n=components.get('N')
        file_z=components.get('Z')
            
        try:
            st_merged=Stream()
            st_merged += read(file_e)
            st_merged += read(file_n)
            st_merged += read(file_z)
            
            t=st_merged[0].stats.starttime
            time_str_iso=t.strftime("%Y-%m-%dT%H%M%S")
            
            output_filename=f"{net}.{sta}.{band_instr}.{time_str_iso}.mseed"
            output_filepath=os.path.join(output_dir, output_filename)
            
            st_merged.write(output_filepath, format="MSEED")
            print(f"{output_filename} succesfully saved!")

        except Exception as e:
            print(f"Error: {net}.{sta}.{band_instr} ({e})")

print("All works completed.")


Processing directory: ./waveforms/251101
KS.CHOA.HG.2025-11-01T000000.mseed succesfully saved!
KS.GKSA.HG.2025-11-01T000000.mseed succesfully saved!
KS.IMSB.HH.2025-11-01T000000.mseed succesfully saved!
KS.JASA.HG.2025-11-01T000000.mseed succesfully saved!
KS.JEU2.EL.2025-11-01T000000.mseed succesfully saved!
KS.JLSA.HG.2025-11-01T000000.mseed succesfully saved!
KS.MALA.HG.2025-11-01T000000.mseed succesfully saved!
KS.NAWB.HH.2025-11-01T000000.mseed succesfully saved!
KS.SIDA.HG.2025-11-01T000000.mseed succesfully saved!
KS.SUCA.HG.2025-11-01T000000.mseed succesfully saved!
KS.TAIA.HG.2025-11-01T000000.mseed succesfully saved!

Processing directory: ./waveforms/251102
KS.CHOA.HG.2025-11-02T000000.mseed succesfully saved!
KS.GKSA.HG.2025-11-02T000000.mseed succesfully saved!
KS.IMSB.HH.2025-11-02T000000.mseed succesfully saved!
KS.JASA.HG.2025-11-02T000000.mseed succesfully saved!
KS.JEU2.EL.2025-11-02T000000.mseed succesfully saved!
KS.JLSA.HG.2025-11-02T000000.mseed succesfully saved